In [1]:
import os
import torch
from bioio import BioImage, Scale
import bioio_tifffile
import numpy as np
import stackview

print (f"Using cuda device: {torch.cuda.is_available()}")

Using cuda device: True


In [2]:
import vistiq
from vistiq.core import ArrayIteratorConfig
from vistiq.preprocess import DoGConfig, DoG
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig
from vistiq.segment.select import RegionFilter, RegionFilterConfig, RangeFilterConfig, RangeFilter
from vistiq.segment.label import MicroSAMSegmenterConfig, MicroSAMSegmenter
from vistiq.segment.postprocess import WatershedConfig, Watershed

# Configure embedding path

In [27]:
embedding_dir = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"
channels = ["dpn_c1"]
z_start = 0 #60
z_end = 127 #70
NB_channel = 0 # zero indexed. if dpn only, use 0
embedding_paths = [os.path.join(embedding_dir, ch_label, f"{z_start}-{z_end}") for ch_label in channels]

# Load image

In [28]:
channel = NB_channel
path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
img = BioImage(path, reader=bioio_tifffile.Reader).data[:,channel,...].squeeze()[z_start:z_end]
img.shape
#stackview.insight(img)

(127, 512, 512)

# Preprocess

In [29]:
dogc = DoGConfig(sigma_low=1.0, sigma_high=12.0)
dimg,_ = DoG(dogc).run(img)

2026-04-29 21:29:38,390 - INFO - Running preprocessor DoG, on stack of type uint8, True
2026-04-29 21:29:38,445 - INFO - Running DoG with config: classname='DoGConfig' package='vistiq.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 tile_shape=None output_type='stack' output_shape=None squeeze=True split_axis=None split_channels=False rename_channel=None normalize=True dtype=None sigma_low=1.0 sigma_high=12.0 mode='reflect'
2026-04-29 21:29:38,445 - INFO - StackProcessor.run: received workers=1 (type: <class 'int'>)
2026-04-29 21:29:38,446 - INFO - Using Parallel with n_jobs=1 for 127 iterations
2026-04-29 21:29:38,447 - INFO - Processing slice, slice.shape=(512, 512)
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
2026-04-29 21:29:38,476 - INFO - Processing slice, slice.shape=(512, 512)
2026-04-29 21:29:38,505 - INFO - Processing slice, slice.shape=(512, 512)
2026

# Define Region Analyzer

In [30]:
vti = ArrayIteratorConfig(slice_def=(-3,2,1))
properties = ["volume", "cross_sectional_area", "area", "aspect_ratio", "bbox"]
ra = RegionAnalyzer(RegionAnalyzerConfig(properties=properties, iterator_config=vti, output_type="dataframe"))

# Define Region Filter

In [42]:
rf = RegionFilter(RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", 
            range=(300,2000)
        )
    ),
    #RangeFilter(
    #    RangeFilterConfig(
    #        attribute="volume",
    #        range=(500,20000)
    #    )
    #),
]))

# Segment

In [43]:
metadata = {
    "scale": Scale(**{
        "Z": 5.0,
        "Y": 1.0,
        "X": 1.0, 
        "T": 1.0, 
        "C": 1.0
    }), 
    "axes": "ZYX"
}
segc = MicroSAMSegmenterConfig(
    embedding_path=embedding_paths[NB_channel], 
    do_regions=True, 
    region_analyzer=ra, 
    region_filter=rf
)
dmask, dlabels, dresults = MicroSAMSegmenter(segc).run(dimg, metadata=metadata)

2026-04-29 21:48:51,710 - INFO - Labeller not provided, using default Labeller with connectivity=1 and region_filter=None
2026-04-29 21:48:51,710 - INFO - Segmenter config: classname=None package=None version=None command_group=None thresholder=None binary_processor=None labeller=Labeller(classname='LabellerConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 tile_shape=None output_type='list' output_shape=None squeeze=True split_axis=None split_channels=False rename_channel=None connectivity=1 region_filter=None) region_analyzer=RegionAnalyzer(classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, 2, 1)) batch_size=10 tile_shape=None output_type='dataframe' output_shape=None squeeze=True split_axis=None split_channel

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'numpy.ndarray'>


2026-04-29 21:49:21,689 - INFO - Running LabelRemover with config: classname='LabelRemoverConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 tile_shape=None output_type='stack' output_shape=None squeeze=False split_axis=None split_channels=False rename_channel=None remap=True
2026-04-29 21:49:21,689 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-04-29 21:49:22,782 - INFO - Finished in state Completed()
2026-04-29 21:49:23,203 - INFO - Finished in state Completed()
2026-04-29 21:49:23,392 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/069f2b50-68a4-721e-8000-7241954d477f/set_state "HTTP/1.1 201 Created"
2026-04-29 21:49:23,937 - INFO - Finished in state Completed()


# Results

In [44]:
dresults.describe()

,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,aspect_ratio,cross_sectional_area,volume
count,252.00000,252.000000,252.000000,252.000000,252.000000,252.000000,203.000000,252.000000,252.000000
mean,60.56746,231.297619,225.488095,66.646825,262.876984,257.285714,0.516918,669.353175,17750.833333
std,29.43423,126.991274,112.408457,29.816542,128.462548,115.603068,0.154613,283.255629,18068.804446
min,0.00000,8.000000,0.000000,3.000000,30.000000,15.000000,0.070477,304.000000,1520.000000
25%,37.00000,118.500000,131.750000,44.000000,150.000000,157.750000,0.407591,429.500000,3790.000000
50%,58.00000,243.000000,240.500000,65.500000,272.500000,271.500000,0.512457,606.000000,10550.000000
75%,85.25000,331.000000,315.250000,92.250000,368.000000,355.250000,0.634094,840.750000,28321.250000
max,125.00000,485.000000,448.000000,127.000000,509.000000,476.000000,0.881442,1498.000000,123940.000000


In [45]:
label_nums = np.unique(dlabels)
for i in range(1, label_nums[-1]+1):
    coords = np.argwhere(dlabels == i)
    top_left = dresults[dresults.index==i][["bbox-0", "bbox-1", "bbox-2"]].to_numpy().squeeze()
    bottom_right = dresults[dresults.index==i][["bbox-3", "bbox-4", "bbox-5"]].to_numpy().squeeze()
    #print (top_left, bottom_right, coords.min(axis=0), coords.max(axis=0)+1)
    assert (np.array_equal(top_left, coords.min(axis=0)))
    assert (np.array_equal(bottom_right, coords.max(axis=0)+1)) 

# Postprocess Labels: Watershed

In [46]:
wsc = WatershedConfig(footprint=(3,15,15), h=1.5, min_distance=1.0, compactness=10)
wlabels = Watershed(wsc).run(dlabels, metadata)

2026-04-29 21:49:50,217 - INFO - metadata['scale']=Scale(T=1.0, C=1.0, Z=5.0, Y=1.0, X=1.0), min_distance=1.0, min_dist_pixel=0
2026-04-29 21:49:50,217 - INFO - footprint=(3, 15, 15)
2026-04-29 21:49:50,722 - INFO - Segmenting mask with intensity value=1; segment values=[0 1], last_label=0
2026-04-29 21:49:50,880 - INFO - Segmenting mask with intensity value=2; segment values=[0 1], last_label=1
2026-04-29 21:49:51,040 - INFO - Segmenting mask with intensity value=3; segment values=[0 1], last_label=2
2026-04-29 21:49:51,198 - INFO - Segmenting mask with intensity value=4; segment values=[0 1], last_label=3
2026-04-29 21:49:51,364 - INFO - Segmenting mask with intensity value=5; segment values=[0 1], last_label=4
2026-04-29 21:49:51,529 - INFO - Segmenting mask with intensity value=6; segment values=[0 1], last_label=5
2026-04-29 21:49:51,684 - INFO - Segmenting mask with intensity value=7; segment values=[0 1], last_label=6
2026-04-29 21:49:51,843 - INFO - Segmenting mask with intensi

In [47]:
import stackview
stackview.slice(np.concatenate([dlabels, wlabels], axis=-1), continuous_update=True)